# STEP 4-B — 배율 강건성: 축소 증강

## 왜 이 실험을 하나

03 에서 입력 해상도를 224 → 384 로 올려 배율 하락을 크게 줄였습니다.

| | 224px | 384px | |
|---|---:|---:|---|
| 1단계 AUROC | 0.8103 | **0.8192** | 하락 17.2% → **9.1%** ✅ |
| 2단계 macro-F1 | 0.5466 | **0.5697** | 하락 29.5% → **25.9%** ❌ |

1단계는 목표(15%) 안에 들어왔지만 **2단계가 남았습니다.** 여기를 마저 잡는 게
이 노트북의 목적입니다.

## 최악 조건은 단계마다 다릅니다

384px 에서 조건별로 뜯어보니 1단계와 2단계가 서로 달랐습니다.

| 조건 | 1단계 | 2단계 macro-F1 |
|---|---:|---:|
| 배율 0.5x (멀리서) | 0.6952 | **0.4066** ← 2단계 최악 |
| 배율 0.71x | 0.7264 | 0.4920 |
| 원본 1x | 0.7474 | 0.5489 |
| 배율 1.41x | 0.7341 | 0.5205 |
| 배율 2x (가까이) | **0.6790** ← 1단계 최악 | 0.4632 |

**1단계는 확대에서, 2단계는 축소에서 무너집니다.**
1단계는 이미 목표를 넘겼으니, 우리가 잡을 **2단계는 축소가 문제**입니다.

## 축소 증강은 224px 에서 실패했는데요

맞습니다. `zoom_both`(축소를 훈련 분포에 넣는 프리셋)를 224px 에서 돌렸다가
27.7% 로 실패했습니다. 그런데 **384px 에서는 조건이 다릅니다.**

교란 검사가 0.5x 를 만드는 방식은 "이미지를 줄이고 반사 패딩으로 채우기" 입니다
(`robust.ZoomView`). 학습 해상도에 따라 남는 정보량이 달라집니다:

| 학습 해상도 | 0.5x 조건에서 실제 내용물 |
|---|---|
| 224px | 짧은 변 **128px** — 질감이 거의 안 남음 |
| **384px** | 짧은 변 **219px** — 224px 원본과 비슷한 정보량 |

224 에서는 **가르칠 정보 자체가 없었습니다.** 384 에서는 남아 있습니다.
같은 프리셋이라도 결과가 달라질 수 있는 이유입니다.

⚠️ **보장은 아닙니다.** 실패하면 "모델링으로는 여기까지" 라는 결론이고,
그때는 촬영 가이드(가까이 찍게 유도)로 넘어갑니다. 그것도 정당한 결말입니다.

## 해상도 카드는 거의 다 썼습니다

03 의 크롭 감사에서 나온 숫자입니다:

```
크롭 짧은 변 중앙값 242px
384px 미만 = 확대해서 씀: 74.6%
```

**이미 74.6% 를 확대해서 쓰고 있습니다.** 512 로 올리면 90% 이상을 늘리게 되는데,
없는 디테일이 생기지는 않습니다. 그래서 이번엔 해상도가 아니라 증강을 봅니다.

## 비교할 것

2단계만 돌립니다 (1단계는 9.1% 로 이미 목표 달성).

| 프리셋 | 하는 일 | 역할 |
|---|---|---|
| `default` | — | 비교 기준 |
| **`zoom_both`** | affine 으로 **실제로 축소** (0.45~1.25배) | **주력** — 최악 조건 직격 |
| `scale_robust` | 확대만 넓힘 (0.80~1.45배) | 대조군 — 확대는 문제가 아님을 확인 |

## 판정 기준 (돌리기 **전에** 정해둡니다)

| 2단계 배율 하락 | 판정 |
|---|---|
| 15% 이하 | ✅ 채택 → 04(백본 비교)로 진행 |
| 줄었지만 15% 초과 | 🤔 방향은 맞음 — 얼마나 더 밀지 상의 |
| 25% 근처 그대로 | ❌ 모델링 종료 → 촬영 가이드로 입력 제한 |

⚠️ **점수가 아니라 하락폭으로 고릅니다.** 증강을 세게 걸면 검증 macro-F1 은 대개 조금
내려갑니다. 그래도 하락폭이 크게 줄면 실사용에는 그쪽이 낫습니다.

⚠️ 같은 설정을 두 번 재서 **20.4%(표본 3,000장) / 25.9%(표본 2,000장)** 가 나왔습니다.
실행·표본에 따라 이만큼 흔들리므로, **셋을 같은 실행에서** 비교하는 게 중요합니다.

⏱️ 3개 × 약 90분 = **약 4시간 30분**. Kaggle 은 `Save & Run All (Commit)` 로 돌리세요.
세션이 끊겨도 다시 돌리면 끝난 학습은 건너뜁니다.


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-21.3"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


---
## 1. 데이터 붙이기


In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages, experiments
from src.config import CLASSES

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# ── 03 에서 확정된 설정 ────────────────────────────────────────
BEST_CROP = "m1.5"     # 2단계 크롭
IMG_SIZE  = 384        # ★ 03 의 해상도 실험에서 채택 (하락 29.5% → 20.4%)
# ⚠️ 03 의 384 실행은 25에폭에 수렴했지만, 증강을 세게 걸면 수렴이 늦어집니다.
#    (1차 증강 실험에서 scale_robust 가 25에폭 전부 best 갱신이었습니다)
#    셋 다 같은 에폭을 줘야 공정한 비교가 되므로 30 으로 맞춥니다.
EPOCHS    = 30

# 03 의 384 실측 (2026-08-21, 표본 2,000장). default 가 여기서 크게 벗어나면
# 데이터·환경이 달라진 것이므로 먼저 원인을 찾으세요.
BASE_384 = {"macro_f1": 0.5697, "scale_drop": 0.259, "worst_at": "배율(0.5x)"}

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")
print(f"{len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")

d = crop.switch_tag(df, BEST_CROP)
s2_all = stages.to_stage2(d)
split.verify(s2_all, fold=0, strict=True)     # 누수 재확인
print(f"2단계 {len(s2_all):,}행")


---
## 2. 증강 프리셋 비교

비교 로직은 `src/experiments.py` 에 있습니다 — 노트북 셀과 달리 `git pull` 로 갱신됩니다.


In [ ]:
# ★ 2단계 최악 조건은 **축소(0.5x)** 입니다 (03 실측). 확대가 아닙니다.
#   그래서 확대만 넓히는 scale_robust 대신 zoom_both 를 주력으로 둡니다.
#   scale_robust 는 "확대는 정말 문제가 아닌가" 를 확인하는 대조군입니다.
PRESETS = ("default", "zoom_both", "scale_robust")

runs = []
for preset in PRESETS:
    runs.append(experiments.train_and_measure(
        s2_all, stage=2, img_size=IMG_SIZE, crop_tag=BEST_CROP,
        device=DEV, epochs=EPOCHS, aug=preset))

verdict = experiments.augmentation_report(runs, baseline="default")


---
## 3. 결과 저장


In [ ]:
import json

W = env.work_root()
(W/"reports").mkdir(parents=True, exist_ok=True)

keep = ("stage", "img_size", "crop_tag", "aug", "exp_name", "epochs", "batch_size",
        "minutes", "best_epoch", "n_epochs", "converged", "score", "score_name",
        "macro_f1", "a6_recall", "scale_drop", "scale_worst", "scale_worst_at")
out = {
    "img_size": IMG_SIZE, "crop": BEST_CROP, "epochs": EPOCHS,
    "baseline_384_from_nb03": BASE_384,
    "runs": [{k: r[k] for k in keep if k in r} for r in runs],
    "verdict": {s: v.get("verdict") for s, v in verdict.items()},
}
(W/"reports"/"step4b_augmentation.json").write_text(
    json.dumps(out, indent=2, ensure_ascii=False))

print("=" * 62)
print(" STEP 4B 결과 (이 블록을 복사해서 공유하세요)")
print("=" * 62)
for r in runs:
    print(f"  {r['aug']:<20} macro-F1 {r['score']:.4f}   "
          f"하락 {r['scale_drop']:.1%}   최악 {r.get('scale_worst_at', '?')}")
print(f"\n  03 의 384 기준 : macro-F1 {BASE_384['macro_f1']:.4f}  "
      f"하락 {BASE_384['scale_drop']:.1%}")
print(f"  판정 : {verdict.get('stage2', {}).get('verdict', '?')}")
print("=" * 62)


---
## 다음 단계

| 이번 판정 | 다음 |
|---|---|
| ✅ 채택 | `src/config.py` 의 기본 증강을 바꾸고 → `04_학습_최신모델_비교` |
| 🤔 개선 | 결과 공유 후 상의 (해상도 512 / 크롭 재설계) |
| ❌ 효과 없음 | 모델링 종료 → `05_평가_보정_GradCAM` + 촬영 가이드 설계 |

⚠️ **`04`(백본 비교)는 2단계 배율 하락이 15% 안에 들어온 뒤에** 돌립니다.
무너지는 기준 위에서 6개 모델을 비교하면 "배율을 가장 잘 읽는 모델" 을 뽑게 되는데,
그건 실사용에서 가장 먼저 무너지는 모델입니다.

📖 [`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md)
